# First agent with smolagents

Same loop, but automated by a framework. Local model with Ollama (`qwen2.5-coder:7b`).

- model: `LiteLLMModel` → local Ollama
- tool: Python function + `@tool`
- harness: `CodeAgent`, actions written as Python code and executed

+ security notes and a scope test (what the agent can or can't do).

In [10]:
import datetime
import pytz
from smolagents import tool

@tool
def get_current_time_in_timezone(timezone: str) -> str:
    """Fetches the current local time in a specified timezone.

    Args:
        timezone: A valid timezone name (e.g., 'America/New_York', 'Europe/Zurich').
    """
    try:
        tz = pytz.timezone(timezone)
        local_time = datetime.datetime.now(tz).strftime("%Y-%m-%d %H:%M:%S")
        return f"The current local time in {timezone} is: {local_time}"
    except Exception as e:
        return f"Error fetching time for timezone '{timezone}': {e}"


In [2]:
# Tool metadata extracted by @tool (what ends up in the system prompt)
print(get_current_time_in_timezone.name)
print(get_current_time_in_timezone.description)
print(get_current_time_in_timezone.inputs)
print(get_current_time_in_timezone.output_type)

get_current_time_in_timezone
Fetches the current local time in a specified timezone.
{'timezone': {'type': 'string', 'description': "A valid timezone name (e.g., 'America/New_York', 'Europe/Zurich')."}}
string


In [3]:
from smolagents import LiteLLMModel

# Local model served by Ollama, accessed through LiteLLM
model = LiteLLMModel(
    model_id="ollama_chat/qwen2.5-coder:7b",
    api_base="http://localhost:11434",
    num_ctx=8192,
    temperature=0.5,
    max_tokens=2096,
)

In [ ]:
# Direct model call to check the Ollama connection
response = model([{"role": "user", "content": [{"type": "text", "text": "The capital of France is"}]}])
print(response.content)

Paris


In [6]:
from smolagents import CodeAgent

agent = CodeAgent(
    model=model,
    tools=[get_current_time_in_timezone],
    max_steps=6,
    verbosity_level=1,
)

In [7]:
agent.run("What time is it right now in Zurich and in Shanghai?")

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ What time is it right now in Zurich and in Shanghai?                                                            │
│                                                                                                                 │
╰─ LiteLLMModel - ollama_chat/qwen2.5-coder:7b ───────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  time_zurich = get_current_time_in_timezone(timezone="Europe/Zurich")                                             
  time_shanghai = get_current_time_in_timezone(timezone="Asia/Shanghai")                                           
  print("Current time in Zurich:", time_zurich)                                                                    
  print("Current time in Shanghai:", time_shanghai)                                                                
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
Current time in Zurich: The current local time in Europe/Zurich is: 2026-11-15 19:11:00
Current time in Shanghai: The current local time in Asia/Shanghai is: 2026-11-16 02:11:00

Out: None

[Step 1: Duration 5.03 seconds| Input tokens: 2,091 | Output tokens: 104]

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 2 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  from datetime import datetime                                                                                    
                                                                                                                   
  # Parse the times                                                                                                
  time_zurich = datetime.strptime(time_zurich.split(': ')[1], '%Y-%m-%d %H:%M:%S')                                 
  time_shanghai = datetime.strptime(time_shanghai.split(': ')[1], '%Y-%m-%d %H:%M:%S')                             
                                                                                                                   
  # Format the times                                                                                               
  formatted_time_zurich = time_zurich.strftime('%H:%M')                                                            
  formatted_time_shanghai = time_shanghai.strftime('%H:%M')                                                        
                                                                                                                   
  final_answer(f"Current time in Zurich: {formatted_time_zurich}\nCurrent time in Shanghai:                        
  {formatted_time_shanghai}")                                                                                      
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: Current time in Zurich: 19:11
Current time in Shanghai: 02:11

[Step 2: Duration 2.99 seconds| Input tokens: 4,479 | Output tokens: 279]

'Current time in Zurich: 19:11\nCurrent time in Shanghai: 02:11'

Testing the agent limit :

In [11]:
# request outside the agent's intended scope
agent.run("List all the files in my home directory.")

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ List all the files in my home directory.                                                                        │
│                                                                                                                 │
╰─ LiteLLMModel - ollama_chat/qwen2.5-coder:7b ───────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  import os                                                                                                        
                                                                                                                   
  home_directory = os.path.expanduser("~")                                                                         
  files = os.listdir(home_directory)                                                                               
  print(files)                                                                                                     
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Warning to user: Code execution failed due to an unauthorized import - Consider passing said import under 
`additional_authorized_imports` when initializing your CodeAgent.

Code execution failed at line 'import os' due to: InterpreterError: Import of os is not allowed. Authorized imports
are: ['collections', 'statistics', 'random', 'math', 're', 'unicodedata', 'stat', 'datetime', 'itertools', 'time', 
'queue']

[Step 1: Duration 4.26 seconds| Input tokens: 2,088 | Output tokens: 81]

... (24 outputs left out)

[Step 6: Duration 2.63 seconds| Input tokens: 17,189 | Output tokens: 859]

Reached max steps.

[Step 7: Duration 2.21 seconds| Input tokens: 19,187 | Output tokens: 933]

'I apologize for the repeated errors. Given the constraints, it seems that it is not possible to list the files in the home directory using the allowed imports. \n\nHowever, if you have access to the file system through a different method, you can manually navigate to the home directory and list the files using the file explorer or command line interface specific to your operating system.'